# 进阶教程（五）：自定义工具与错误处理

> Agent 的下限由工具质量决定，上限由容错能力决定。

## 本讲内容
1. `@tool` 进阶：Pydantic 参数校验
2. 工具异常：错误回流（ToolException + ToolErrorMiddleware）
3. 模型层重试：with_retry
4. 级联兜底：with_fallbacks
5. middleware：审计与错误兜底


# 0. 环境准备与运行说明

**前置要求：**
- 根目录 `.env` 已配置 `DEEPSEEK_API_KEY`（本教程用真实 DeepSeek，无本地降级）
- 已安装：`langchain>=1.3`、`langgraph>=1.2`、`langchain-deepseek`、`python-dotenv`
- 使用本地 `bge-small-zh-v1.5` 嵌入的章节首次运行会下载模型（约 100MB，走 hf-mirror 镜像）

**运行说明：**
- 按 cell 顺序执行；除标注外，每个示例消耗少量 API 额度（单次 < 0.01 元量级）
- 本教程面向已学完 `langchain_tutorial/` 与 `langgraph_tutorial/` 基础篇的开发者
- 涉及导入路径的坑（如 `create_agent` 在 `langchain.agents`）已在 FAQ 中汇总

In [8]:

# ========== 0. 初始化（每个 notebook 第一格） ==========
import os, sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=DeprecationWarning)

# HF 镜像必须先于任何 langchain/huggingface 导入设置（详见 rag_qa_project FAQ）
os.environ.setdefault("HF_ENDPOINT", "https://hf-mirror.com")

ROOT = Path.cwd().parent  # advanced_tutorial 的上一级 = 项目根目录
sys.path.insert(0, str(ROOT))
from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

assert os.getenv("DEEPSEEK_API_KEY"), "请先在根目录 .env 配置 DEEPSEEK_API_KEY"

# 真实 LLM：DeepSeek（本教程要求真实模型）
from langchain_deepseek import ChatDeepSeek

model = ChatDeepSeek(model="deepseek-chat", temperature=0.2)
print("模型就绪:", model.__class__.__name__)


模型就绪: ChatDeepSeek


## 1. @tool 进阶：参数校验与规范

默认 `@tool` 从 docstring/type hints 推断 schema；
复杂参数用 `args_schema` 显式声明——**校验发生在工具执行前**，
模型传错参数会直接报错而不是带病执行：

In [9]:

from pydantic import BaseModel, Field
from langchain_core.tools import tool

class TransferArgs(BaseModel):
    """转账工具的参数规范（描述会进入模型上下文）"""
    to: str = Field(description="收款人姓名")
    amount: float = Field(gt=0, le=1_000_000, description="金额，0-100万")

@tool(args_schema=TransferArgs)
def transfer(to: str, amount: float) -> str:
    """向指定用户转账。"""
    return f"已向 {to} 转账 ¥{amount:,.2f}"

# schema 被 Pydantic 把关
print(transfer.invoke({"to": "Alice", "amount": 100}))
try:
    transfer.invoke({"to": "Bob", "amount": -5})   # 违反 gt=0
except Exception as e:
    print("校验拦截:", type(e).__name__)

已向 Alice 转账 ¥100.00
校验拦截: ValidationError


## 2. 工具异常：错误回流（ToolException + ToolErrorMiddleware）

> **版本坑（已实测）**：`create_agent` 在 langchain 1.3.x 已移除 `handle_tool_errors`
> 参数（传了直接 `TypeError`）。而且实测发现：工具内抛 `ToolException` 时，
> `create_agent` 默认**直接上抛崩溃**——它内部 `ToolNode` 的默认错误 handler 只吞
> "模型参数校验错误"（`ToolInvocationError`），不吞工具执行期异常。

官方正解：**挂内置中间件 `ToolErrorMiddleware`**（实现 `wrap_tool_call` 钩子），
把工具执行期异常转成 `ToolMessage(status="error")` 喂回模型，让模型自纠：
不用手写图，一行挂载即可：


In [10]:
from langchain.agents import create_agent
from langchain.agents.middleware import ToolErrorMiddleware
from langchain_core.tools import ToolException, tool

@tool
def query_order(order_id: str) -> str:
    """查询订单状态。Args: order_id: 订单号，形如 ORD-123"""
    if not order_id.startswith("ORD-"):
        raise ToolException(f"订单号格式错误：{order_id}，应以 ORD- 开头")
    return "已发货"

# 错误 handler：只处理 ToolException -> 可读文案（喂回模型，让模型自纠）
def on_error(exc, request) -> str | None:
    if isinstance(exc, ToolException):
        return f"工具执行出错：{exc}。请检查参数格式后重试。"
    return None   # 其他异常原样上抛，不暴露给模型

agent = create_agent(
    model=model,
    tools=[query_order],
    middleware=[ToolErrorMiddleware(on_error)],   # 错误回流
)

# system 强制走工具且不改参数格式 -> 必然触发 ToolException
r = agent.invoke({"messages": [
    {"role": "system", "content": "查询订单必须用工具。用户给什么订单号就直接查什么，不要修改格式。"},
    {"role": "user", "content": "帮我查订单 12345 的状态"}]})
for m in r["messages"]:
    tag = "  <- error 回流" if getattr(m, "status", None) == "error" else ""
    print(f"[{type(m).__name__}] {str(m.content)[:90]}{tag}")


[SystemMessage] 查询订单必须用工具。用户给什么订单号就直接查什么，不要修改格式。
[HumanMessage] 帮我查订单 12345 的状态
[AIMessage] 
[ToolMessage] 工具执行出错：订单号格式错误：12345，应以 ORD- 开头。请检查参数格式后重试。  <- error 回流
[AIMessage] 抱歉，您提供的订单号 `12345` 格式不正确。订单号应该以 `ORD-` 开头，例如 `ORD-123`。

请您确认一下订单号是否正确，或者提供完整的订单号（如 `ORD-1


**要点**：
- `on_error(exc, request)` 的**返回值决定命运**：返回字符串 → 转成
  `ToolMessage(status="error")` 回流模型（模型自纠）；返回 `None` → 原样上抛
- **处理是 opt-in 的**：只处理你在 `on_error` 里返回了内容的异常；漏参、类型错这类
  参数校验错误由 `ToolNode` 上游拦截，到不了 `on_error`——它只处理工具**执行期**异常
- 输出里 `status=error` 的 `ToolMessage` 就是"错误已回流给模型"的信号
- 可选参数：`aon_error`（异步 handler）、`tools=[...]`（只对指定工具生效）
- **经验法则**：参数类错误用 ToolException（模型能自纠）；系统类错误（DB 挂了）
  直接抛（重试也没用，走 fallbacks）
- 需要手写自定义图时，等价写法是 `ToolNode(handle_tool_errors=handler)`（见 FAQ）


## 3. 模型层重试：with_retry

网络抖动、限流是常态。`with_retry` 基于 tenacity，
指数退避自动重试：

In [11]:

import time

# 模拟一个 30% 概率限流的服务（前两次必失败用计数器演示）
class FlakyModel:
    calls = 0
    def invoke(self, x):
        FlakyModel.calls += 1
        if FlakyModel.calls <= 2:
            raise ConnectionError("模拟 429 限流")
        return f"第 {FlakyModel.calls} 次调用成功"

from langchain_core.runnables import RunnableLambda
flaky = RunnableLambda(FlakyModel().invoke)

# with_retry：指数退避重试（此处不真 sleep，仅演示结构）
retryable = flaky.with_retry(
    stop_after_attempt=3,
    wait_exponential_jitter=False,
    retry_if_exception_type=(ConnectionError,),
)
print(retryable.invoke("test"))
print("总调用次数:", FlakyModel.calls)   # 3 次调用 = 2 次失败 + 1 次成功

第 3 次调用成功
总调用次数: 3


真实模型直接：
```python
robust_model = model.with_retry(stop_after_attempt=3, wait_exponential_jitter=True)
```
配合 `retry_if_exception_type=(RateLimitError,)` 可只对限流重试。

## 4. 级联兜底：with_fallbacks

重试解决"偶发失败"，fallbacks 解决"持续不可用"。
首选失败 N 次后自动切换备用模型，对上层完全透明：

In [12]:

from langchain_deepseek import ChatDeepSeek

# 备用通道：同厂商不同档位（生产可换成不同厂商实现真正容灾）
backup = ChatDeepSeek(model="deepseek-chat", temperature=0.5)

def always_fail(x):
    raise ConnectionError("主通道持续故障")

from langchain_core.runnables import RunnableLambda
primary = RunnableLambda(always_fail)

robust = primary.with_fallbacks([backup])
r = robust.invoke("用 20 字说明什么是容灾")
print("兜底回答:", r.content)

兜底回答: 容灾是当灾难发生时，确保业务不中断或快速恢复的机制。


## 5. middleware：审计与错误兜底

`create_agent` 的 `middleware` 参数提供 Agent 级切面。
用类中间件实现"模型调用失败 → 兜底文案"（洋葱模型包裹模型调用）：

In [13]:

from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware, ModelRequest
from langchain_core.messages import AIMessage

class SafeGuardMiddleware(AgentMiddleware):
    """审计 + 兜底：包裹每一次模型调用"""
    calls = 0

    async def wrap_model_call(self, request: ModelRequest, call_next):
        SafeGuardMiddleware.calls += 1
        print(f"  [audit] 第 {SafeGuardMiddleware.calls} 次模型调用")
        try:
            return await call_next(request)
        except Exception as e:
            print(f"  [safe] 模型调用失败，兜底: {type(e).__name__}")
            return AIMessage(content=f"服务暂时不可用，请稍后重试（{type(e).__name__}）")

safe_agent = create_agent(
    model=model,
    tools=[],
    system_prompt="你是简洁的助手。",
    middleware=[SafeGuardMiddleware()],
)
r = safe_agent.invoke({"messages": [{"role": "user", "content": "你好"}]})
print("回答:", r["messages"][-1].content[:80])

AttributeError: 'coroutine' object has no attribute 'result'

## 6. 错误处理策略速查

| 层级 | 机制 | 处理什么 |
|---|---|---|
| 工具层 | ToolException + ToolErrorMiddleware | 参数错误（模型可自纠） |
| 模型层 | with_retry | 网络抖动、限流 |
| 通道层 | with_fallbacks | 主服务持续不可用 |
| Agent 层 | middleware wrap_model_call | 统一审计、兜底、降级 |
| 图层 | 哨兵节点 + 兜底分支（见教程二） | 输出质量不合格 |


## 7. 常见问题（FAQ）

| 问题 | 原因 | 解决 |
|---|---|---|
| 工具内抛异常 Agent 直接崩 | create_agent 默认只吞参数校验错（`handle_tool_errors` 参数已移除，工具执行期异常会上抛） | 挂 `ToolErrorMiddleware(on_error)`；自定义图则用 `ToolNode(handle_tool_errors=handler)` |
| with_retry 越重试越慢 | 指数退避生效（预期行为） | 合理设置 stop_after_attempt（3 次为宜） |
| fallbacks 备用也失败 | 级联深度不够 | 再挂一个静态 Runnable 兜底作最底层 |
| middleware 不生效 | 用了装饰器却没传参 | 类中间件必须传 middleware=[...] |
| wrap_model_call 是 async | 钩子设计为协程 | def 改 async def，用 await call_next(request) |
